# 🚀 Direct access to EarthScope S3 cloud storage: s3fs


**Author:** Yiyu Ni  
**Goal:** This notebook demonstrates how to access earthscope S3 cloud stoage and load miniSEED data.  

---

In [1]:
import io
import obspy
import s3fs

from parameters import ES_S3_ACCESS_POINT
from earthscope_sdk import EarthScopeClient

## 1. Get temporary credential through EarthScope Client

In [2]:
client = EarthScopeClient()
cred = client.user.get_aws_credentials()
print(f"Credential expires at {cred.expiration} UTC")

Credential expires at 2025-10-14 06:45:34+00:00 UTC


## 2. Prepare S3 access
The EarthScope credential object **cred** provides a temporary AWS access key ID, access key, and session token, all valid for an hour.

In [3]:
fs = s3fs.S3FileSystem(
    key=cred.aws_access_key_id,
    secret=cred.aws_secret_access_key,
    token=cred.aws_session_token,
)

### 3. Access the bucket through the S3 access point
The **ES_S3_ACCESS_POINT** provides the entry point to the S3 bucket. Instead of directly accessing an S3 bucket, additional authentication setting at finer level could be configured through this layer. The access point is defined [here](./parameters.py).

With s3fs, we can browse the bucket with Linux-style command, e.g., `fs.ls`. 

In [4]:
["/".join(i.split("/")[1:]) for i in fs.ls(f"{ES_S3_ACCESS_POINT}/miniseed/CC/2006/001/")]

['miniseed/CC/2006/001/JRO.CC.2006.001#1',
 'miniseed/CC/2006/001/NED.CC.2006.001#1',
 'miniseed/CC/2006/001/RAFT.CC.2006.001#2',
 'miniseed/CC/2006/001/SEP.CC.2006.001#1',
 'miniseed/CC/2006/001/STD.CC.2006.001#1']

As shown here, we see files from each station is saved as individual files, or **objects** given the context that we are accessing data in a **S3 bucket**.

---
Here, we can use `fs.read_bytes` to read one file. Finally, we parse the bytes into `obspy.Stream`.

In [5]:
bytes = fs.read_bytes(f"{ES_S3_ACCESS_POINT}/miniseed/CC/2006/001/STD.CC.2006.001#1")
print(len(bytes))

34823680


In [6]:
buff = io.BytesIO(bytes)

In [7]:
obspy.read(buff)

5 Trace(s) in Stream:
CC.STD..BHE | 2006-01-01T00:00:00.000000Z - 2006-01-01T20:05:14.980000Z | 50.0 Hz, 3615750 samples
CC.STD..BHE | 2006-01-01T20:05:20.020000Z - 2006-01-01T23:59:59.980000Z | 50.0 Hz, 703999 samples
CC.STD..BHN | 2006-01-01T00:00:00.000000Z - 2006-01-01T23:59:59.980000Z | 50.0 Hz, 4320000 samples
CC.STD..BHZ | 2006-01-01T00:00:00.000000Z - 2006-01-01T20:05:14.980000Z | 50.0 Hz, 3615750 samples
CC.STD..BHZ | 2006-01-01T20:05:20.020000Z - 2006-01-01T23:59:59.980000Z | 50.0 Hz, 703999 samples